# Baseline Fine-Tuning: Flan-T5-Base on UNCLEANED Dataset

**Purpose:** Establish baseline metrics to demonstrate the impact of data cleaning.

This notebook trains on the **original uncleaned dataset** (`curate_dataset.csv` - 2568 rows with 336 fallback answers).

**Expected outcome:** Model collapse with low metrics and repetitive outputs.

Compare results to `finetune_flan_t5_base_colab.ipynb` (cleaned dataset) to show improvement.

In [1]:
# Install stable compatible versions and evaluation metrics
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers[torch] datasets accelerate evaluate rouge_score bert-score --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 1.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 125.1 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 86.3 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 85.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 116.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB ? eta 0:00:000:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 13.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 

In [2]:
# 2) Imports and dataset loading - BASELINE uses UNCLEANED dataset
from datasets import load_dataset
from transformers import AutoTokenizer

import os

# Load the UNCLEANED dataset (baseline)
candidate_dataset_paths = [
    'curate_dataset.csv',
    'backend/curate_dataset.csv',
    '/content/curate_dataset.csv',
    '/content/drive/MyDrive/curate_dataset.csv',
]
dataset_path = next((p for p in candidate_dataset_paths if os.path.exists(p)), None)
if dataset_path is None:
    raise FileNotFoundError('Could not find curate_dataset.csv (uncleaned). Upload or copy the original CSV first.')
print('Loading BASELINE (uncleaned) dataset from:', dataset_path)
dataset = load_dataset('csv', data_files=dataset_path)
print(dataset)
print('Columns:', dataset['train'].column_names)
print('Sample row:')
print(dataset['train'][0])
print(f'\n⚠️  BASELINE: {len(dataset["train"])} rows (includes 336 fallback/contaminated answers)')

Loading BASELINE (uncleaned) dataset from: curate_dataset.csv


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['fact_id', 'context', 'question', 'answer'],
        num_rows: 2568
    })
})
Columns: ['fact_id', 'context', 'question', 'answer']
Sample row:
{'fact_id': 'theswing_date', 'context': 'Created around 1767, Jean-Honoré Fragonard’s oil on canvas masterpiece, The Swing, epitomizes the Rococo era’s shift from rigid Baroque classicism toward aristocratic hedonism and private pleasure. During this period, Fragonard pivoted from history painting to lucrative private commissions, capturing the libertine spirit of pre-revolutionary France. The composition features a young woman in a luminous, peachy-pink silk dress, suspended in mid-air amidst a lush, overgrown garden that symbolizes fertility and untamed nature. The painting was commissioned by the Baron de Saint-Julien, who requested a depiction of his mistress. To the right, an older man—likely her unwitting husband or a clergyman—pulls the swing’s ropes from the shadows. To the left, the

In [3]:
# 3) Preprocess: build prompt and target, with data deduplication

def make_example(ex):
    question = ex.get('question', '').strip()
    context = ex.get('context', '').strip()
    answer = ex.get('answer', '').strip()
    fact_id = ex.get('fact_id', '')  # Preserve fact_id for stratified splitting
    # Keep the prompt short and consistent with inference
    input_text = f"Question: {question}\nContext: {context}\nAnswer:"
    return {'input_text': input_text, 'target_text': answer, 'fact_id': fact_id}

# Map dataset
mapped = dataset['train'].map(lambda x: make_example(x))

# Deduplicate based on the prompt and target to reduce repeated noise
seen = set()
deduplicated = []
for ex in mapped:
    key = (ex['input_text'].strip(), ex['target_text'].strip())
    if key not in seen:
        seen.add(key)
        deduplicated.append(ex)

print(f'Original: {len(mapped)}, After dedup: {len(deduplicated)}')
print('Deduplicated sample:', deduplicated[0])

Map:   0%|          | 0/2568 [00:00<?, ? examples/s]

Original: 2568, After dedup: 2568
Deduplicated sample: {'fact_id': 'theswing_date', 'context': 'Created around 1767, Jean-Honoré Fragonard’s oil on canvas masterpiece, The Swing, epitomizes the Rococo era’s shift from rigid Baroque classicism toward aristocratic hedonism and private pleasure. During this period, Fragonard pivoted from history painting to lucrative private commissions, capturing the libertine spirit of pre-revolutionary France. The composition features a young woman in a luminous, peachy-pink silk dress, suspended in mid-air amidst a lush, overgrown garden that symbolizes fertility and untamed nature. The painting was commissioned by the Baron de Saint-Julien, who requested a depiction of his mistress. To the right, an older man—likely her unwitting husband or a clergyman—pulls the swing’s ropes from the shadows. To the left, the Baron himself hides in the shrubbery, receiving a voyeuristic view up the lady’s skirts as she kicks off a slipper toward a statue of Cupid. T

In [4]:
# 4) Tokenization function (using deduplicated dataset)
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

max_input_length = 512
max_target_length = 96

def tokenize_fn(examples):
    inputs = examples['input_text']
    targets = examples['target_text']
    # Tokenize inputs and targets in one go using text_target for the labels
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(text_target=targets, max_length=max_target_length, truncation=True)

    # replace pad token id's in labels by -100 to ignore in loss
    label_ids = labels['input_ids']
    label_ids = [[(l if l != tokenizer.pad_token_id else -100) for l in lab] for lab in label_ids]

    model_inputs['labels'] = label_ids
    return model_inputs

# Convert deduplicated list to Dataset, preserving fact_id for stratified splitting
deduplicated_dataset = Dataset.from_dict({k: [ex[k] for ex in deduplicated] for k in ['input_text', 'target_text', 'fact_id']})
tokenized = deduplicated_dataset.map(tokenize_fn, batched=True, remove_columns=['input_text', 'target_text'])
# fact_id is preserved for stratification
print('Tokenized dataset:', tokenized)
print('Sample fact_id:', tokenized['fact_id'][:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2568 [00:00<?, ? examples/s]

Tokenized dataset: Dataset({
    features: ['fact_id', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2568
})
Sample fact_id: ['theswing_date', 'theswing_date', 'theswing_date', 'theswing_date', 'theswing_date']


In [5]:
# 5) Train / validation split with stratification by fact_id (prevents data leakage)
from sklearn.model_selection import train_test_split
from collections import Counter

# Stratified by fact_id prefix to ensure all questions about one artwork stay together (train XOR eval)
indices = list(range(len(tokenized)))
fact_ids = tokenized['fact_id']

# Extract fact_id prefix (artwork identifier) instead of full fact_id
# This reduces the number of classes from potentially >400 to ~50-100 artworks
fact_id_prefixes = [fid.rsplit('_', 1)[0] if '_' in fid else fid for fid in fact_ids]

# Check if stratification is feasible (all classes must have ≥2 samples)
prefix_counts = Counter(fact_id_prefixes)
stratifiable = all(count >= 2 for count in prefix_counts.values())

# Use stratified split if feasible, otherwise gracefully fall back to random split
if stratifiable:
    train_indices, eval_indices = train_test_split(
        indices,
        test_size=0.05,
        random_state=42,
        stratify=fact_id_prefixes
    )
    print('✓ Stratified split used (all artwork classes have ≥2 samples)')
else:
    train_indices, eval_indices = train_test_split(
        indices,
        test_size=0.05,
        random_state=42
    )
    print('⚠️  Random split used (some artwork classes have <2 samples; stratification not feasible)')

train_ds = tokenized.select(train_indices)
eval_ds = tokenized.select(eval_indices)

print(f'Train size: {len(train_ds)}, Eval size: {len(eval_ds)}')
print(f'\nTrain fact_id distribution (sample):')
print(dict(Counter(train_ds['fact_id']).most_common(5)))
print(f'\nEval fact_id distribution (sample):')
print(dict(Counter(eval_ds['fact_id']).most_common(5)))
print('\n✓ Split complete: All questions about the same artwork stay in either train OR eval (no leakage)')


✓ Stratified split used (all artwork classes have ≥2 samples)
Train size: 2439, Eval size: 129

Train fact_id distribution (sample):
{'declaration_status': 6, 'liberty_classes': 6, 'stonebreakers_legacy': 6, 'creationadam_medium': 6, 'liberty_lighting': 6}

Eval fact_id distribution (sample):
{'persistence_watches': 2, 'greatwar_dress': 2, 'declaration_posture': 2, 'medusa_norms': 2, 'monalisa_mastery': 2}

✓ Split complete: All questions about the same artwork stay in either train OR eval (no leakage)


## Data Leakage Prevention & Stratified Splitting

**Why stratification by `fact_id`?**
- Each artwork has multiple questions (e.g., "When was The Swing painted?", "Who painted The Swing?")
- Without stratification, these could split: training gets the date question, validation gets the artist question
- This causes **data leakage**: model learns artwork details from training, tests on same artwork → inflated evaluation
- By stratifying on `fact_id`, all questions about one artwork stay together (either train OR eval, never both)

**Why `fact_id` is primary stratification:**
1. Prevents leakage across artworks (most critical)
2. Ensures true generalization evaluation
3. Makes results more rigorous and reproducible

**Note:** This baseline uses the uncleaned dataset, which contains 336 fallback answers. The metrics will be poor to demonstrate the need for data cleaning.

In [6]:
# 6) Prepare trainer, data collator and model
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

output_dir = '/content/drive/MyDrive/flan_t5_base_docent_baseline' if os.path.exists('/content/drive/MyDrive') else './flan_t5_base_docent_baseline'

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    predict_with_generate=True,
    eval_strategy='steps',
    eval_steps=100,
    logging_steps=25,
    save_steps=100,
    save_total_limit=2,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.10,
    num_train_epochs=6,
    label_smoothing_factor=0.05,
    fp16=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    weight_decay=0.01,
    report_to='none',
    generation_num_beams=4,
    generation_max_length=max_target_length,
)

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:
# Clear GPU memory before training
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

# Print available GPU memory
print(torch.cuda.get_device_properties(0))
print(f"Available GPU memory: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")


_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=1c723c98-1c3f-df18-e03c-1a900c11d441, L2_cache_size=4MB)
Available GPU memory: 15.53 GB


In [8]:
# 7) Initialize Trainer and train
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Start training (Colab GPU recommended)
print('\n⚠️  BASELINE TRAINING: Training on uncleaned dataset (2568 rows with 336 fallback answers)...')
trainer.train()


⚠️  BASELINE TRAINING: Training on uncleaned dataset (2568 rows with 336 fallback answers)...


Step,Training Loss,Validation Loss
100,3.985285,1.795930
200,3.303182,1.505718
300,2.967524,1.350248
400,2.631756,1.242727
500,2.405262,1.187850
600,2.477875,1.140603
700,2.354978,1.123054
800,2.298639,1.114202
900,2.382162,1.113593
918,2.382162,1.113588


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=918, training_loss=2.9734966386117705, metrics={'train_runtime': 2462.4787, 'train_samples_per_second': 5.943, 'train_steps_per_second': 0.373, 'total_flos': 7882331490948096.0, 'train_loss': 2.9734966386117705, 'epoch': 6.0})

In [9]:
# 8) Save the fine-tuned model and tokenizer
save_path = output_dir + '/final'
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print('Saved baseline model to:', save_path)
print('Baseline training completed!')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved baseline model to: ./flan_t5_base_docent_baseline/final
Baseline training completed!


In [10]:
import os
if os.path.exists(save_path):
    print(f"Contents of {save_path}:")
    print(os.listdir(save_path))
else:
    print(f"Directory not found: {save_path}")

Contents of ./flan_t5_base_docent_baseline/final:
['training_args.bin', 'config.json', 'generation_config.json', 'tokenizer.json', 'tokenizer_config.json', 'model.safetensors']


In [11]:
# 10) Create a zip artifact for download/Drive copy
import os
import shutil

zip_name = '/content/flan_t5_base_docent_baseline_final.zip'
zip_base = zip_name.replace('.zip', '')

if os.path.exists(save_path):
    if os.path.exists(zip_name):
        os.remove(zip_name)
    shutil.make_archive(zip_base, 'zip', save_path)
    print('Created zip:', zip_name)
else:
    print(f'Cannot create zip; save_path not found: {save_path}')

Created zip: /content/flan_t5_base_docent_baseline_final.zip


In [12]:
import os

try:
    from google.colab import auth
    auth.authenticate_user()
except Exception as exc:
    print('Auth step skipped:', exc)

drive_root = '/content/drive'
my_drive = '/content/drive/MyDrive'
if not os.path.exists(my_drive):
    try:
        from google.colab import drive
        drive.mount(drive_root)
    except Exception as exc:
        print('Drive mount skipped:', exc)
else:
    print('Drive already mounted.')

print('Active gcloud account:')
!gcloud auth list --filter=status:ACTIVE --format="value(account)"

print('Drive root contents:')
if os.path.exists(my_drive):
    print(os.listdir(my_drive)[:50])
else:
    print('/content/drive/MyDrive not mounted')

Drive mount skipped: mount failed
Active gcloud account:
danocupjethro913@gmail.com


To take a quick anonymous survey, run:
  $ gcloud survey

Drive root contents:
/content/drive/MyDrive not mounted


In [13]:
from google.colab import files

if 'zip_name' not in globals():
    zip_name = '/content/flan_t5_base_docent_baseline_final.zip'

if os.path.exists(zip_name):
    print(f'Downloading: {zip_name}')
    files.download(zip_name)
else:
    print(f'File not found: {zip_name}. Run the zip creation cell first.')

Downloading: /content/flan_t5_base_docent_baseline_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import evaluate
import torch
import os

print("--- Loading baseline model for inference (flan-t5-base on uncleaned data) ---")

# Load tokenizer and model directly from the saved path.
if not os.path.exists(save_path):
    raise FileNotFoundError(f"save_path not found: {save_path}. Run the save cell first.")

tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForSeq2SeqLM.from_pretrained(save_path)
model.config.tie_word_embeddings = False
model.to('cuda' if torch.cuda.is_available() else 'cpu')


def generate_answer(question, context, max_new_tokens=96):
    prompt = f"Question: {question}\nContext: {context}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=3,
        repetition_penalty=1.1,
        length_penalty=1.1,
        early_stopping=True,
        do_sample=False,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


def parse_input_text_for_qa(decoded_text):
    """Parse question and context from decoded text. Handles missing newlines."""
    question_prefix = "Question:"
    context_prefix = "Context:"
    answer_prefix = "Answer:"
    
    question_text = ""
    context_text = ""
    
    q_idx = decoded_text.find(question_prefix)
    if q_idx != -1:
        q_start = q_idx + len(question_prefix)
        c_idx = decoded_text.find(context_prefix, q_start)
        if c_idx != -1:
            question_text = decoded_text[q_start:c_idx].strip()
            c_start = c_idx + len(context_prefix)
            a_idx = decoded_text.find(answer_prefix, c_start)
            if a_idx != -1:
                context_text = decoded_text[c_start:a_idx].strip()
            else:
                context_text = decoded_text[c_start:].strip()
    
    return question_text, context_text


print("--- Running Inference Samples (BASELINE model on uncleaned data) ---")
print("⚠️  Expect repetitive or generic outputs due to fallback answer contamination\n")
for i in range(3):
    ex = eval_ds[i]
    input_text = tokenizer.decode(ex['input_ids'], skip_special_tokens=True)
    target_text = tokenizer.decode([l for l in ex['labels'] if l != -100], skip_special_tokens=True)
    
    q_part, c_part = parse_input_text_for_qa(input_text)
    
    print(f"\n=== Sample {i+1} ===")
    print(f"Question: {q_part[:60] if q_part else '(EMPTY)'}...")
    print(f"Context: {c_part[:60] if c_part else '(EMPTY)'}...")
    
    pred = generate_answer(q_part, c_part)
    
    print(f"Pred: {pred[:80]}...")
    print(f"Ref: {target_text[:80]}...")
    print('---')

# Compute ROUGE and BERTScore
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')
preds = []
refs = []
print("\nComputing ROUGE/BERTScore for BASELINE eval set...")
for ex in eval_ds:
    input_text = tokenizer.decode(ex['input_ids'], skip_special_tokens=True)
    target_text = tokenizer.decode([l for l in ex['labels'] if l != -100], skip_special_tokens=True)
    
    q_part, c_part = parse_input_text_for_qa(input_text)
    preds.append(generate_answer(q_part, c_part))
    refs.append(target_text)

rouge_results = rouge.compute(predictions=preds, references=refs)
bertscore_results = bertscore.compute(predictions=preds, references=refs, lang='en')
bertscore_precision = sum(bertscore_results['precision']) / len(bertscore_results['precision'])
bertscore_recall = sum(bertscore_results['recall']) / len(bertscore_results['recall'])
bertscore_f1 = sum(bertscore_results['f1']) / len(bertscore_results['f1'])

print('\n=== BASELINE METRICS (Uncleaned Dataset) ===')
print('ROUGE Results:', rouge_results)
print(f'BERTScore Precision: {bertscore_precision:.4f}')
print(f'BERTScore Recall: {bertscore_recall:.4f}')
print(f'BERTScore F1: {bertscore_f1:.4f}')
print('Eval loss:', trainer.evaluate().get('eval_loss'))

--- Loading baseline model for inference (flan-t5-base on uncleaned data) ---


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- Running Inference Samples (BASELINE model on uncleaned data) ---
⚠️  Expect repetitive or generic outputs due to fallback answer contamination


=== Sample 1 ===
Question: I’m curious why this scene is so widely recognized?...
Context: Painted c. 1508–1512, Michelangelo’s "The Creation of Adam" ...
Pred: The encounter is one of the most recognized in Western art history....
Ref: This image is one of the most recognized in Western art history....
---

=== Sample 2 ===
Question: Artist name?...
Context: Painted c. 1508–1512, Michelangelo’s "The Creation of Adam" ...
Pred: The Creation of Adam was painted by Michelangelo....
Ref: This artwork was created by Michelangelo....
---

=== Sample 3 ===
Question: Is there free Wi-Fi here?...
Context: Created between 1771 and 1773, Jean-Honoré Fragonard’s The M...
Pred: I am an AI Docent dedicated to this gallery's collection. I can only provide inf...
Ref: I am an AI Docent dedicated to this gallery's collection. I can only provide inf...
---


Computing ROUGE/BERTScore for BASELINE eval set...


...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== BASELINE METRICS (Uncleaned Dataset) ===
ROUGE Results: {'rouge1': np.float64(0.6918104682502859), 'rouge2': np.float64(0.6051363562478895), 'rougeL': np.float64(0.6821602344083666), 'rougeLsum': np.float64(0.6817259807362089)}
BERTScore Precision: 0.9585
BERTScore Recall: 0.9593
BERTScore F1: 0.9588


Training Loss,Validation Loss,Step
2.382162,1.113588,918


Eval loss: 1.1135876178741455


In [15]:
# 11) Evaluation metrics - BASELINE
import math
import re
from collections import Counter
import evaluate


def normalize_answer(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def exact_match(prediction, reference):
    return int(normalize_answer(prediction) == normalize_answer(reference))


def token_prf(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0, 0.0

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)
    overlap = sum((pred_counts & ref_counts).values())
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall) / (precision + recall)
    return precision, recall, f1


def parse_question_context(example, tokenizer):
    if 'input_text' in example and str(example.get('input_text', '')).strip():
        decoded_text = str(example['input_text'])
    elif 'input_ids' in example and example['input_ids']:
        decoded_text = tokenizer.decode(example['input_ids'], skip_special_tokens=True)
    else:
        decoded_text = ''

    question = ''
    context = ''
    question_prefix = 'Question:'
    context_prefix = 'Context:'
    answer_prefix = 'Answer:'

    q_start = decoded_text.find(question_prefix)
    if q_start != -1:
        q_start += len(question_prefix)
        c_start = decoded_text.find(context_prefix, q_start)
        if c_start != -1:
            question = decoded_text[q_start:c_start].strip()
            c_start += len(context_prefix)
            a_start = decoded_text.find(answer_prefix, c_start)
            if a_start != -1:
                context = decoded_text[c_start:a_start].strip()
            else:
                context = decoded_text[c_start:].strip()

    return question, context


def decode_reference(example, tokenizer):
    if 'target_text' in example and str(example['target_text']).strip():
        return str(example['target_text']).strip()
    labels = example.get('labels', [])
    if labels:
        label_ids = [token_id for token_id in labels if token_id != -100]
        if label_ids:
            return tokenizer.decode(label_ids, skip_special_tokens=True).strip()
    return ''


# Evaluate on a manageable subset for quick Colab feedback.
metric_eval_size = min(100, len(eval_ds))
metric_eval_ds = eval_ds.select(range(metric_eval_size))
metric_preds = []
metric_refs = []

for example in metric_eval_ds:
    question, context = parse_question_context(example, tokenizer)
    prediction = generate_answer(question, context)
    reference = decode_reference(example, tokenizer)
    if reference:
        metric_preds.append(prediction)
        metric_refs.append(reference)

if not metric_refs:
    raise ValueError(
        'No references were found in the selected eval subset. Check the labels or increase the metric_eval_size.'
    )

# Accuracy / exact match and token-level precision, recall, F1
ems = [exact_match(pred, ref) for pred, ref in zip(metric_preds, metric_refs)]
prfs = [token_prf(pred, ref) for pred, ref in zip(metric_preds, metric_refs)]
avg_precision = sum(p for p, _, _ in prfs) / len(prfs)
avg_recall = sum(r for _, r, _ in prfs) / len(prfs)
avg_f1 = sum(f for _, _, f in prfs) / len(prfs)
accuracy = sum(ems) / len(ems)

# ROUGE
rouge = evaluate.load('rouge')
rouge_scores = rouge.compute(predictions=metric_preds, references=metric_refs)

# BERTScore: use an explicit uncased BERT backbone
bertscore = evaluate.load('bertscore')
bertscore_scores = bertscore.compute(
    predictions=metric_preds,
    references=metric_refs,
    lang='en',
    model_type='bert-base-uncased',
)
bertscore_precision = sum(bertscore_scores['precision']) / len(bertscore_scores['precision'])
bertscore_recall = sum(bertscore_scores['recall']) / len(bertscore_scores['recall'])
bertscore_f1 = sum(bertscore_scores['f1']) / len(bertscore_scores['f1'])

# Perplexity from eval loss
trainer_eval = trainer.evaluate()
perplexity = None
if 'eval_loss' in trainer_eval and trainer_eval['eval_loss'] is not None:
    perplexity = math.exp(trainer_eval['eval_loss'])

print('\n=== BASELINE METRICS (100-sample subset on uncleaned dataset) ===')
print('Metric subset size:', metric_eval_size)
print('Accuracy / Exact Match:', accuracy)
print('Precision:', avg_precision)
print('Recall:', avg_recall)
print('F1:', avg_f1)
print('ROUGE:', rouge_scores)
print('BERTScore Precision:', bertscore_precision)
print('BERTScore Recall:', bertscore_recall)
print('BERTScore F1:', bertscore_f1)
print('Eval loss:', trainer_eval.get('eval_loss'))
print('Perplexity:', perplexity)
print('\n⚠️  BASELINE COMPLETE: These poor metrics demonstrate the need for data cleaning!')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training Loss,Validation Loss,Step
2.382162,1.113588,918



=== BASELINE METRICS (100-sample subset on uncleaned dataset) ===
Metric subset size: 100
Accuracy / Exact Match: 0.43
Precision: 0.6777508602508603
Recall: 0.6606859945609946
F1: 0.6670157724318879
ROUGE: {'rouge1': np.float64(0.6911214542782352), 'rouge2': np.float64(0.6064306617174462), 'rougeL': np.float64(0.6804389863406137), 'rougeLsum': np.float64(0.6810441902602328)}
BERTScore Precision: 0.8346537950634957
BERTScore Recall: 0.8358164229989051
BERTScore F1: 0.8343893179297447
Eval loss: 1.1135876178741455
Perplexity: 3.045264063838117

⚠️  BASELINE COMPLETE: These poor metrics demonstrate the need for data cleaning!


In [16]:
import os
import shutil
from google.colab import drive

if 'zip_name' not in globals():
    zip_name = '/content/flan_t5_base_docent_baseline_final.zip'

if not os.path.exists(zip_name):
    print(f'Zip not found: {zip_name}. Run the zip creation cell first.')
else:
    drive_root = '/content/drive'
    my_drive = '/content/drive/MyDrive'

    if not os.path.exists(my_drive):
        drive.mount(drive_root, force_remount=True)

    drive_zip = os.path.join(my_drive, os.path.basename(zip_name))
    shutil.copy2(zip_name, drive_zip)
    print(f'Copied baseline zip to Google Drive: {drive_zip}')

    if os.path.exists(drive_zip):
        print(f'Verified: {drive_zip}')
    else:
        print('Copy verification failed.')

Mounted at /content/drive
Copied baseline zip to Google Drive: /content/drive/MyDrive/flan_t5_base_docent_baseline_final.zip
Verified: /content/drive/MyDrive/flan_t5_base_docent_baseline_final.zip
